## Installing Python Packages

In [1]:
!pip install pylatexenc
!pip install qiskit
!pip install qiskit-aer
!pip install opencv-python

## Importing Python Packages

In [2]:
import os
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from numpy import asarray
from pathlib import Path
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.circuit.library import RYGate
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import Permutation, UnitaryGate
from qiskit.circuit.library import MCXGate
from qiskit.quantum_info import Operator
import time
import math
from skimage.measure import shannon_entropy
from skimage.morphology import skeletonize

## Simulation Setup

In [3]:
backend_sim = AerSimulator(method="statevector")

def run_simulation_statevector(qc,backend_sim):
  compiled_circuit = transpile(qc, backend_sim,optimization_level=1)
  result = backend_sim.run(compiled_circuit).result()
  statevector = result.get_statevector(compiled_circuit)
  statevector = np.real(np.asarray(statevector))

  return statevector

## Image Pre-Processing

In [4]:
def load_and_display_image(image_path, display_img = True, invert = False):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    if invert:
        image = 255 - image
    if display_img:
        plt.imshow(image,cmap='gray')
        plt.title('Image')
        plt.axis('off')
        plt.show()

    return image

def resize_image(image, new_size=(64, 64)):
    resized_image = cv2.resize(image, new_size, interpolation=cv2.INTER_NEAREST)
    return resized_image

def crop_image(image, new_size=(64, 64)):
    croped_image = image[0:new_size[0], 0:new_size[1]]
    return croped_image

def save_image(image, file_name = None):
    plt.imsave(file_name, image, cmap='gray')

def plot_image(image, title):
    size_img = np.shape(image)
    title = title + " " + str(size_img)

    plt.imshow(image, extent=[0,image.shape[0], image.shape[1],0,], cmap='gray')
    plt.xticks(range(0,image.shape[0],2))
    plt.yticks(range(0,image.shape[1],2))
    plt.axis('off')
    plt.title(title)
    plt.show()

def plot_images(images, titles):
    size_img = np.shape(images[0])
    f, axarr = plt.subplots(1,len(images), figsize=(15,3*len(images)))

    for i in range(len(images)):
        title = titles[i] + " " + str(size_img)
        axarr[i].set_title(title)
        axarr[i].set_xticks(range(0,images[i].shape[0],10))
        axarr[i].set_yticks(range(0,images[i].shape[1],10))
        axarr[i].imshow(images[i], cmap='gray')
        axarr[i].axis('off')

    plt.show()

## Flexible Representation of Quantum Images

In [5]:
def frqi(image, measure = True):
    image_array=np.asarray(image).flatten()
    qubit_num = int(np.log2(len(image_array)))

    pixel_qubit = QuantumRegister(1, 'pixel qubit')
    pos_qubits = QuantumRegister(qubit_num, 'position qubit')
    classical_bits = ClassicalRegister(qubit_num + 1, 'classical')
    qc = QuantumCircuit(pixel_qubit,pos_qubits, classical_bits)

    image_array = image_array / 255.0 ### Normalizing grayscale values
    theta_values = [math.asin(image_array[k]) for k in range(len(image_array))]

    enc_state = [] ### Form: psi = cos(t)|i>|0> + sin(t)|i>|1>
    for k in range(0,2**(qubit_num+1),2):
      enc_state.append(np.cos(theta_values[int(k/2)]))
      enc_state.append(np.sin(theta_values[int(k/2)]))

    enc_state = np.asarray(enc_state)
    enc_state = enc_state/(2**(qubit_num/2))

    qc.initialize(enc_state,range(qubit_num+1))

    if measure:
        qc.measure_all(add_bits=False)
        return qc
    else:
        return qc, pos_qubits, pixel_qubit, classical_bits, qubit_num

def frqi_gradient(img, grad_type = None):

  qed_bit = ClassicalRegister(1, 'qhed_bit')

  qc, position_qubits, qed_qubit, classical_bits , qubit_num = frqi(img, measure=False)
  qc.add_register(qed_bit)

  # Build a new circuit with registers in the order: psi_reg, qed_qubit, then classical registers.
  # Measure qhed into classical bit c
  qc.measure(qed_qubit[0], qed_bit[0])

  # Conditionally apply X if measurement outcome == 1
  with qc.if_test((qed_bit[0], 1)):
      qc.x(qed_qubit[0])

  if grad_type == 'lag1' or grad_type == 'lag2':
    qc.h(qed_qubit[0])
    #################################
    qc.barrier()
    ### Permutation operator
    qc.x(qed_qubit[0])
    for i in range(qubit_num):
        if i == 0:
            qc.cx(qed_qubit[0], position_qubits[0])
        else:
            # Define controls: always include qhed[0] and the first i position qubits.
            controls = [qed_qubit[0]] + list(position_qubits[:i])
            target = position_qubits[i]
            # Append an MCX gate with the appropriate number of controls.
            qc.append(MCXGate(len(controls)), qargs=controls + [target])
    #################################
    qc.barrier()
    #################################
    if grad_type == 'lag2':
      qc.x(qed_qubit[0])
      #################################
      qc.barrier()
      ### Permutation operator
      qc.x(qed_qubit[0])
      for i in range(qubit_num):
          if i == 0:
              qc.cx(qed_qubit[0], position_qubits[0])
          else:
              # Define controls: always include qhed[0] and the first i position qubits.
              controls = [qed_qubit[0]] + list(position_qubits[:i])
              target = position_qubits[i]
              # Append an MCX gate with the appropriate number of controls.
              qc.append(MCXGate(len(controls)), qargs=controls + [target])
      #################################
    #################################
    qc.barrier()
    qc.h(qed_qubit[0])
  else: print("Error! Please provide kernel type.")
  qc.save_statevector()

  return qc, grad_type

def retrieve_grad_frqi(statevector, grad_type = 'None', Sobel = False):
    n = int((np.log2(len(statevector))-1)//2) # Get n from the first dict key length

    ### grad = [c0-c2, c1-c3, c2-c4, ----]
    grad = [statevector[i] for i in range(1,len(statevector)+1,2)]


    if grad_type == 'lag1':
      grad = np.asarray(grad)
      img_grad = grad.reshape((2**n,2**n))

      return img_grad

    elif grad_type == 'lag2':
      img_grad = grad[-1:] + grad[:-1] ### Cyclic shift to the right

      img_grad = np.asarray(img_grad)
      img_grad = img_grad.reshape((2**n,2**n))

      #### construction of grad Sobel kernel
      if Sobel:
        sobel = np.zeros((2**n,2**n))
        for i in range(1,2**n-1):
          for j in range(1,2**n-1):
            sobel[i,j] = img_grad[i,j-1] + 2*img_grad[i,j] + img_grad[i,j+1]

        return sobel
      #######################################

      else: return img_grad

    else: print("Error! Please provide kernel type.")

## Connecting to QHCD folder

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
### Urban Dataset
# dataset_path = Path('./drive/MyDrive/QHCD/UrbanDataset/Images')
# ### Animal images
# dataset_path = Path('./drive/MyDrive/QHCD/Animals')
### Other images
dataset_path = Path('./drive/MyDrive/QHCD/Edge Detection Img')

pixel_size = [256, 512, 1024]
pixel_size_num = len(pixel_size)

file_count = 0
for entry in dataset_path.iterdir():
      if entry.is_file() and entry.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
        print("Image File: ", entry.name)
        file_count = file_count + 1

print("\n Number of Test Images: ", file_count)

Image File:  03-35028.png
Image File:  04-0896x4.png
Image File:  05-WIREFRAME-2.png
Image File:  06-elephant_3.png
Image File:  08-ADE20K-1C.png
Image File:  12-cameraman.png
Image File:  15-P1020854.png
Image File:  21-00065305.png

 Number of Test Images:  8


## Edge Detection using QHED
### Enoding Method: FRQI

In [ ]:
edge_density = np.zeros(file_count)
thickness_ratio = np.zeros(file_count)
edge_fragments = np.zeros(file_count)
entropy_val = np.zeros(file_count)

edge_density_QHED = np.zeros(file_count)
thickness_ratio_QHED = np.zeros(file_count)
edge_fragments_QHED = np.zeros(file_count)
entropy_val_QHED = np.zeros(file_count)

pixel_size = 512
# threshold = [0.375, 0.2, 0.3, 0.2, 0.35] ### Animal Dataset (Eagle,Lion,Deer,Dog,Cat)
threshold = [0.15,0.275,0.15,0.125,0.25,0.15,0.15,0.175]
# threshold = [0.001]*file_count

i_file = 0
# Iterate over all files in the dataset
for entry in dataset_path.iterdir():
    # Optionally, filter to only process image files based on their extension
    if entry.is_file() and entry.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
        print("\n Image File: ", entry.name)

        img_file = entry.name
        # img_name = img_file.split('.')[0]  # Split by '.' and take the first part
        # otherwise use entry.stem (name of the file without extension)

        image_path = dataset_path / img_file
        test_image = load_and_display_image(image_path,display_img = False)

        print("\n --------------------------------------------- \n")

        print("\n Image Size: ", pixel_size, "x", pixel_size)
        image = resize_image(test_image, (pixel_size,pixel_size))
        plot_image(image,entry.name)

        ##################### SOBEL Kernel #####################
        ########## Encoding ##########
        print("\n Performing FRQI Sobel Gradient Encoding ----- ")

        start_time = time.perf_counter()

        print("\n Performing FRQI Sobel Gradient Encoding along x-direction ----- ")
        qc_frqi_x, frqi_grad_type = frqi_gradient(image,grad_type='lag2')

        print("\n Performing FRQI Sobel Gradient Encoding along y-direction ----- ")
        qc_frqi_y, frqi_grad_type = frqi_gradient(image.T,grad_type='lag2')
        # print(qc_FRQI.draw())

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"FRQI Sobel Gradient Encoding Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ##### Decoding
        print("\n Retreiving FRQI Sobel Gradient ----- ")

        start_time = time.perf_counter()

        statevector_x = run_simulation_statevector(qc_frqi_x,backend_sim)
        statevector_y = run_simulation_statevector(qc_frqi_y,backend_sim)

        # print(statevector_x)
        # print(statevector_y)

        grad_x = retrieve_grad_frqi(statevector_x,frqi_grad_type, Sobel = True)
        grad_y = retrieve_grad_frqi(statevector_y,frqi_grad_type, Sobel = True)

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"FRQI Sobel Gradient Retrieval Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ### Gradient Magnitude
        sobel_mag = np.sqrt(grad_x**2 + (grad_y.T)**2)

        ### Normalizing gradient
        sobel_mag = cv2.normalize(sobel_mag, None, 0, 1.0, cv2.NORM_MINMAX)

        # threshold = 0.3  # You can tune this value
        edges = (sobel_mag > threshold[i_file]).astype(np.uint8)
        # Making edge thickness = 1
        edges_thin = skeletonize((edges > 0).astype(np.uint8))

        ####################################################################################
        ####################################################################################
        ##################### QHED #####################
        ########## Encoding ##########
        print("\n Performing FRQI QHED Gradient Encoding ----- ")

        start_time = time.perf_counter()

        print("\n Performing FRQI QHED Gradient Encoding along x-direction ----- ")
        qc_frqi_x_QHED, frqi_grad_type_QHED = frqi_gradient(image,grad_type='lag1')

        print("\n Performing FRQI QHED Gradient Encoding along y-direction ----- ")
        qc_frqi_y_QHED, frqi_grad_type_QHED = frqi_gradient(image.T,grad_type='lag1')
        # print(qc_FRQI.draw())

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"FRQI QHED Gradient Encoding Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ##### Decoding
        print("\n Retreiving FRQI QHED Gradient ----- ")

        start_time = time.perf_counter()

        statevector_x_QHED = run_simulation_statevector(qc_frqi_x_QHED,backend_sim)
        statevector_y_QHED = run_simulation_statevector(qc_frqi_y_QHED,backend_sim)

        # print(statevector_x)
        # print(statevector_y)

        grad_x_QHED = retrieve_grad_frqi(statevector_x_QHED,frqi_grad_type_QHED)
        grad_y_QHED = retrieve_grad_frqi(statevector_y_QHED,frqi_grad_type_QHED)

        end_time = time.perf_counter()
        elapsed_seconds = end_time - start_time
        print(f"FRQI QHED Gradient Retrieval Complete! Elapsed time: {elapsed_seconds:.6f} seconds\n")

        ### Gradient Magnitude
        sobel_mag_QHED = np.sqrt(grad_x_QHED**2 + (grad_y_QHED.T)**2)

        ### Normalizing gradient
        sobel_mag_QHED = cv2.normalize(sobel_mag_QHED, None, 0, 1.0, cv2.NORM_MINMAX)

        # threshold = 0.3  # You can tune this value
        edges_QHED = (sobel_mag_QHED > threshold[i_file]).astype(np.uint8)
        # Making edge thickness = 1
        edges_thin_QHED = skeletonize((edges_QHED > 0).astype(np.uint8))

        ####################################################################################
        ####################################################################################
        print("Edge Detection using Sobel Kernel")
        plot_images([image, edges, edges_thin],["Original image", "Edge","Thin edges"])
        print("Edge Detection using QHED")
        plot_images([image, edges_QHED, edges_thin_QHED],["Original image", "Edge","Thin edges"])

        ####################################################################################
        ####################################################################################
        ### Calculating Performance Metrics Sobel Kernel
        print("\n Performance Metrics (Sobel Kernel) ----- ")

        edge_density[i_file] = np.sum(edges_thin) / edges_thin.size
        print(f"Edge Density: {edge_density[i_file]:.4f}")

        skeleton = skeletonize(edges_thin)
        thickness_ratio[i_file] = np.sum(edges_thin) / np.sum(skeleton)
        print(f"Average Edge Thickness: {thickness_ratio[i_file]:.2f}")

        contours, _ = cv2.findContours(edges_thin.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        edge_fragments[i_file] = len(contours)
        print(f"Number of edge fragments: {edge_fragments[i_file]}")

        entropy_val[i_file] = shannon_entropy(edges_thin)
        print(f"Edge Map Entropy: {entropy_val[i_file]:.2f}")

        ### Calculating Performance Metrics QHED
        print("\n Performance Metrics (QHED) ----- ")

        edge_density_QHED[i_file] = np.sum(edges_thin_QHED) / edges_thin.size
        print(f"Edge Density: {edge_density_QHED[i_file]:.4f}")

        skeleton = skeletonize(edges_thin_QHED)
        thickness_ratio_QHED[i_file] = np.sum(edges_thin_QHED) / np.sum(skeleton)
        print(f"Average Edge Thickness: {thickness_ratio_QHED[i_file]:.2f}")

        contours, _ = cv2.findContours(edges_thin_QHED.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        edge_fragments_QHED[i_file] = len(contours)
        print(f"Number of edge fragments: {edge_fragments_QHED[i_file]}")

        entropy_val_QHED[i_file] = shannon_entropy(edges_thin_QHED)
        print(f"Edge Map Entropy: {entropy_val_QHED[i_file]:.2f}")

        ####################################################################################
        ####################################################################################

        ### Saving reconstructed image
        # img_output_name = f"FRQI_{entry.stem}_Sobel{entry.suffix}"
        img_output_name = f"FRQI_{entry.stem}_Sobel.jpg"
        output_path = dataset_path.parent / 'Edge Detection using Sobel' / img_output_name
        save_image(edges, file_name = output_path)

        img_output_name = f"FRQI_{entry.stem}_SobelThin.jpg"
        output_path = dataset_path.parent / 'Edge Detection using Sobel' / img_output_name
        save_image(edges_thin, file_name = output_path)

        img_output_name = f"FRQI_{entry.stem}_QHED.jpg"
        output_path = dataset_path.parent / 'Edge Detection using QHED' / img_output_name
        save_image(edges_QHED, file_name = output_path)

        img_output_name = f"FRQI_{entry.stem}_QHEDThin.jpg"
        output_path = dataset_path.parent / 'Edge Detection using QHED' / img_output_name
        save_image(edges_thin_QHED, file_name = output_path)

        i_file = i_file + 1